# Language Detection

Build a multi-class classifier using traditional NLP to classify the language of the customer's message.

In [1]:
!pip install datasets scikit-learn pandas numpy matplotlib seaborn


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\LEGION\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [3]:
import sys
!{sys.executable} -m pip install datasets

  Using cached datasets-5.0.1-py3-none-any.whl.metadata (23 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
Using cached datasets-5.0.1-py3-none-any.whl (559 kB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
   ---------------------------------------- 0.0/27.8 MB ? eta -:--:--
   - -------------------------------------- 0.8/27.8 MB 4.2 MB/s eta 0:00:07
   -- ------------------------------------- 1.6/27.8 MB 4.0 MB/s eta 0:00:07
   --- ------------------------------------ 2.4/27.8 MB 3.9 MB/s eta 0:00:07
   ---- ----------------------------------- 3.1/27.8 MB 3.9 MB/s eta 0:00:07
   ----- ---------------------------------- 3.9/27.8 MB 3.9 MB/s eta 0:00:07
   ------ --------------------------------- 4.7/27.8 MB 3.9 MB/s eta 0:00:06
   ------- -------------------------------- 5.5/27.8 MB 3.9 MB/s eta 0:00:06
   --------- -----------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


## Load Dataset
Using `papluca/language-identification`

In [4]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset('papluca/language-identification')
train_df = pd.DataFrame(dataset['train'])
val_df = pd.DataFrame(dataset['validation'])
test_df = pd.DataFrame(dataset['test'])
train_df.head()

c:\Users\LEGION\anaconda3\envs\dental_xray_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,labels,text
0,pt,"os chefes de defesa da estónia, letónia, lituâ..."
1,bg,размерът на хоризонталната мрежа може да бъде ...
2,zh,很好，以前从不去评价，不知道浪费了多少积分，现在知道积分可以换钱，就要好好评价了，后来我就把...
3,th,สำหรับ ของเก่า ที่ จริงจัง ลอง honeychurch ...
4,ru,Он увеличил давление .


## Preprocessing & Vectorization

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train = le.fit_transform(train_df['labels'])
y_val = le.transform(val_df['labels'])
y_test = le.transform(test_df['labels'])

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_df['text'])
X_val = vectorizer.transform(val_df['text'])
X_test = vectorizer.transform(test_df['text'])

## Model Training (Logistic Regression)

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_val_pred = model.predict(X_val)
print('Validation Accuracy:', accuracy_score(y_val, y_val_pred))
print(classification_report(y_val, y_val_pred, target_names=le.classes_))

Validation Accuracy: 0.9051
              precision    recall  f1-score   support

          ar       1.00      0.88      0.94       500
          bg       0.99      0.96      0.97       500
          de       1.00      0.98      0.99       500
          el       1.00      0.99      1.00       500
          en       0.97      0.98      0.98       500
          es       0.98      0.99      0.99       500
          fr       1.00      0.98      0.99       500
          hi       1.00      0.94      0.97       500
          it       0.99      0.99      0.99       500
          ja       0.81      0.09      0.17       500
          nl       0.99      0.99      0.99       500
          pl       0.96      0.90      0.93       500
          pt       0.98      0.99      0.99       500
          ru       0.99      0.91      0.95       500
          sw       0.99      0.94      0.97       500
          th       1.00      0.69      0.82       500
          tr       0.97      0.95      0.96       500

## Evaluation on Test Set

In [7]:
y_test_pred = model.predict(X_test)
print('Test Accuracy:', accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred, target_names=le.classes_))

Test Accuracy: 0.9067
              precision    recall  f1-score   support

          ar       1.00      0.92      0.96       500
          bg       0.99      0.97      0.98       500
          de       1.00      0.98      0.99       500
          el       1.00      0.99      0.99       500
          en       0.99      0.99      0.99       500
          es       0.98      0.99      0.99       500
          fr       0.99      0.99      0.99       500
          hi       1.00      0.95      0.97       500
          it       0.99      0.96      0.98       500
          ja       0.71      0.09      0.16       500
          nl       1.00      0.96      0.98       500
          pl       0.96      0.88      0.92       500
          pt       0.97      0.95      0.96       500
          ru       0.99      0.93      0.96       500
          sw       0.99      0.97      0.98       500
          th       1.00      0.71      0.83       500
          tr       0.97      0.93      0.95       500
     

## Save Model

In [8]:
import joblib
import os

os.makedirs('models', exist_ok=True)
joblib.dump(model, 'models/language_detector.pkl')
joblib.dump(vectorizer, 'models/language_vectorizer.pkl')
joblib.dump(le, 'models/language_encoder.pkl')
print('Model saved to models/')

Model saved to models/
